# RustGIS：臺灣地震空間探究
Python 3.12+；依序執行。合成資料不是觀測紀錄。

In [ ]:
%pip install 'rustgis[jupyter]'

## 1 世界地圖：圖層、分級設色與輸出

In [ ]:
import rustgis
from pathlib import Path
out = Path("outputs")
out.mkdir(exist_ok=True)
s = rustgis.Session(sample=True)
print(s.layer_names())
s["Countries"].style("graduated", field="pop_est")
s.to_png(str(out / "world.png"), width=1200, height=800)
s

## 2 地震分布：經度在前、緯度在後；深度 km 與規模是不同變數。

In [ ]:
import json
sample = {"type":"FeatureCollection","features":[{"type":"Feature","geometry":{"type":"Point","coordinates":[121.6,24.1]},"properties":{"id":"S1","mag":6.5,"depth_km":15,"source":"synthetic"}},{"type":"Feature","geometry":{"type":"Point","coordinates":[121.8,24.3]},"properties":{"id":"S2","mag":5.2,"depth_km":22,"source":"synthetic"}},{"type":"Feature","geometry":{"type":"Point","coordinates":[121.2,23.1]},"properties":{"id":"S3","mag":5.8,"depth_km":10,"source":"synthetic"}},{"type":"Feature","geometry":{"type":"Point","coordinates":[120.5,23.5]},"properties":{"id":"S4","mag":4.2,"depth_km":12,"source":"synthetic"}},{"type":"Feature","geometry":{"type":"Point","coordinates":[122,25]},"properties":{"id":"S5","mag":5.1,"depth_km":85,"source":"synthetic"}},{"type":"Feature","geometry":{"type":"Point","coordinates":[121.5,24.7]},"properties":{"id":"S6","mag":4.8,"depth_km":65,"source":"synthetic"}}]}
s = rustgis.Session()
q = s.add_geojson(sample, name="Synthetic earthquakes")
q.style("proportional", field="mag")
s.zoom_to(q)
print(q.count, q.fields)
assert q.count == 6
s.to_png(str(out / "earthquakes.png"), width=1200, height=800)
s

## 3 POE：預測 30、50、100 km 環域的差異。環域不是震度或預警範圍。

In [ ]:
print(s.describe("buffer"))
for radius in (30, 50, 100):
    b = s.run_tool("buffer", input=q, distance=radius, units="Kilometers", output=f"Buffer {radius} km")
    b.set_transparency(65)
    print(radius, b.count)
s.zoom_to(b)
s.to_png(str(out / "buffers.png"), width=1200, height=800)
s.save(str(out / "earthquakes.rgis"))
s

## 4 真實資料：USGS 2024 年 4 月臺灣附近 M≥4 地震，UTC；規模類型可能不同，不等於 CWA ML。

In [ ]:
from urllib.request import urlopen
from urllib.parse import urlencode
params = dict(format="geojson", starttime="2024-04-01", endtime="2024-05-01", minlatitude=21, maxlatitude=26, minlongitude=119, maxlongitude=123, minmagnitude=4, orderby="time-asc")
url = "https://earthquake.usgs.gov/fdsnws/event/1/query?" + urlencode(params)
with urlopen(url, timeout=60) as response:
    raw = json.load(response)
features = []
for f in raw["features"]:
    lon, lat, depth = f["geometry"]["coordinates"]
    p = f["properties"]
    features.append(dict(type="Feature", geometry=dict(type="Point", coordinates=[lon,lat]), properties=dict(id=f["id"], mag=p["mag"], depth_km=depth, mag_type=p.get("magType"), time_ms=p["time"])))
assert features, "No events returned"
real = dict(type="FeatureCollection", features=features)
(out / "usgs.geojson").write_text(json.dumps(real), encoding="utf-8")
r = rustgis.Session()
layer = r.add_geojson(real, name="USGS earthquakes")
layer.style("graduated", field="depth_km")
r.zoom_to(layer)
print(layer.count)
r.to_png(str(out / "usgs_depth.png"), width=1200, height=800)
r

## 5 互動介面：需網路連線。若 widget 無法顯示，可使用前面的靜態圖或 WebGIS 手動載入 GeoJSON。

In [ ]:
m = rustgis.Map()
m.add_geojson(sample, name="Synthetic earthquakes")
m["Synthetic earthquakes"].style("proportional", field="mag")
m.zoom_to(m["Synthetic earthquakes"])
m

## 6 CER：主張東部是否較集中？證據使用事件數與圖層；推理須討論研究範圍、目錄完整性與時間窗。請 AI 解釋 buffer 參數，再用 describe 與實際結果核對。不得把環域當作災害風險。